# Stage 1 — Twitter US Airline Sentiment dataset audit

This notebook audits the approved dataset without splitting data, fitting a vocabulary or building/training a model. Future work is a custom PyTorch 1D CNN versus GRU using random embeddings and weights. No pretrained assets, Keras or TensorFlow are used.

The assignment brief, marking scheme, lecturer materials, disclosure guidance and earlier audit are missing from the workspace. Compliance with them is pending. This notebook is a working audit, not an assignment report. See `docs/requirements_and_assistance.md` for the factual assistance record.

Run using `.venv/Scripts/python.exe src/run_notebook.py` from the project root. The runner starts a fresh kernel using that exact interpreter and saves executed outputs here. The audit implementation is in `src/audit.py`; it is part of this deliverable.

In [1]:
from pathlib import Path
import sys, json, platform, hashlib
from datetime import datetime, timezone
import importlib.metadata as metadata
import pandas as pd
import psutil
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'data/original/Tweets.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src/audit.py').exists(), 'Run within this project.'
assert Path(sys.prefix).resolve() == (ROOT / '.venv').resolve(), 'Use the project environment.'
sys.path.insert(0, str(ROOT / 'src'))
from audit import run_audit, normalise, linked_groups, EXPECTED_SHA256

# Simple CPU tensor/autograd compatibility check. This is not model training.
x = torch.tensor([1.0, 2.0], requires_grad=True)
(x.square().sum()).backward()
assert torch.equal(x.grad, torch.tensor([2.0, 4.0]))
# pandas may expose a read-only view. Give PyTorch its own writable array.
assert torch.equal(torch.from_numpy(pd.Series([1., 2.]).to_numpy(copy=True)), torch.tensor([1., 2.], dtype=torch.float64))
mem = psutil.virtual_memory()
environment = {
    'captured_utc': datetime.now(timezone.utc).isoformat(),
    'scope': 'Local Codex desktop execution host with the user workspace on C:; not a remote Linux runtime.',
    'platform': platform.platform(), 'architecture': platform.machine(),
    'python': sys.version, 'executable': sys.executable, 'prefix': sys.prefix,
    'ram_total_GiB': round(mem.total / 2**30, 3), 'ram_available_GiB': round(mem.available / 2**30, 3),
    'torch_version': torch.__version__, 'torch_cuda_build': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'xpu_available': bool(hasattr(torch, 'xpu') and torch.xpu.is_available()),
    'cpu_tensor_autograd_and_numpy_smoke_check': 'passed',
    'packages': {p: metadata.version(p) for p in ['pandas','numpy','nbformat','nbclient','ipykernel','jupyterlab','psutil','scikit-learn','matplotlib','torch']},
}
(ROOT / 'results/environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
display(environment)
display(json.loads((ROOT / 'results/host_windows.json').read_text(encoding='utf-8-sig')))
display(json.loads((ROOT / 'results/host_gpu.json').read_text(encoding='utf-8-sig')))

{'captured_utc': '2026-09-13T00:43:06.509309+00:00',
 'scope': 'Local Codex desktop execution host with the user workspace on C:; not a remote Linux runtime.',
 'platform': 'Windows-11-10.0.26200-SP0',
 'architecture': 'AMD64',
 'python': '3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]',
 'executable': '<local-path-redacted> 'prefix': '<local-path-redacted> 'ram_total_GiB': 15.701,
 'ram_available_GiB': 2.027,
 'torch_version': '2.14.0+cpu',
 'torch_cuda_build': None,
 'cuda_available': False,
 'xpu_available': False,
 'cpu_tensor_autograd_and_numpy_smoke_check': 'passed',
 'packages': {'pandas': '3.0.5',
  'numpy': '2.5.3',
  'nbformat': '5.11.1',
  'nbclient': '0.11.0',
  'ipykernel': '7.3.0',
  'jupyterlab': '4.6.3',
  'psutil': '7.2.2',
  'scikit-learn': '1.9.1',
  'matplotlib': '3.11.2',
  'torch': '2.14.0+cpu'}}

{'Caption': 'Microsoft Windows 11 Home',
 'Version': '10.0.26200',
 'OSArchitecture': '64-bit',
 'TotalVisibleMemorySize': 16463352,
 'FreePhysicalMemory': 1831916}

{'Name': 'Intel(R) Iris(R) Xe Graphics',
 'DriverVersion': '31.0.101.4255',
 'AdapterRAM': 1073741824}

## Provenance and preservation

Source: [CrowdFlower Twitter US Airline Sentiment on Kaggle](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment).
Downloaded directly from `https://www.kaggle.com/api/v1/datasets/download/crowdflower/twitter-airline-sentiment`. No mirror was used. The ZIP is retained; `Tweets.csv` was extracted byte-for-byte. Checksums establish the exact snapshot used here, not an independent publisher signature. The endpoint is mutable and no Kaggle version identifier was captured in the download record.

See `data/original/provenance.json`. All columns are loaded as strings with automatic NA inference disabled, so tweet IDs never pass through floating point and literal text such as "NA" is preserved. Empty and whitespace-only text are checked explicitly. `source_row` is the 1-based data record number (CSV records can span physical lines).

In [2]:
display(json.loads((ROOT / 'data/original/provenance.json').read_text(encoding='utf-8')))
raw = pd.read_csv(ROOT / 'data/original/Tweets.csv', dtype='string', keep_default_na=False, encoding='utf-8')
assert raw.tweet_id.map(lambda value: isinstance(value, str)).all()
assert hashlib.sha256((ROOT / 'data/original/Tweets.csv').read_bytes()).hexdigest() == EXPECTED_SHA256
print('Starting records:', len(raw))
display(raw.dtypes.to_frame('dtype'))

{'dataset_page': 'https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment',
 'download_url': 'https://www.kaggle.com/api/v1/datasets/download/crowdflower/twitter-airline-sentiment',
 'provider': 'Kaggle; publisher account crowdflower',
 'mirror_used': False,
 'download_date_local': '2026-09-13',
 'timezone': 'Asia/Singapore',
 'snapshot_version': 'Not captured by the download; hashes identify local bytes',
 'archive_members': ['Tweets.csv', 'database.sqlite'],
 'extraction': 'Tweets.csv read directly from ZIP without reserialisation; database.sqlite retained inside ZIP only',
 'files': {'Tweets.csv': {'bytes': 3421431,
   'sha256': 'ea94b23f41892b290dec3330bb8cf9cb6b8bc669eaae5f3a84c40f7b0de8f15e'},
  'twitter-airline-sentiment.zip': {'bytes': 2678605,
   'sha256': 'c0dbee48cac32110a607430dc5c1941a1728383adef20a893ded91adbbba2de2'}}}

Starting records: 14640


,dtype
tweet_id,string
airline_sentiment,string
airline_sentiment_confidence,string
negativereason,string
negativereason_confidence,string
airline,string
airline_sentiment_gold,string
name,string
negativereason_gold,string
retweet_count,string


## Exact normalisation and exclusion policy

The duplicate key applies, in order: (1) Unicode NFC canonical composition, (2) replace each run matching Python Unicode `\s+` with one ASCII space, (3) strip leading/trailing whitespace. No lowercasing, HTML unescaping, stemming, tokenisation, stopword removal, URL/mention removal or punctuation removal is performed. Negation, emojis, variation selectors, zero-width joiners, punctuation and case remain. The original `text` column is unchanged. This key is for auditing, not a final model preprocessing decision.

Removals occur sequentially:

1. Exclude missing/blank normalised text.
2. Exclude labels other than exactly `negative`, `neutral`, `positive`; do not repair label spellings.
3. Connect valid rows sharing a nonblank tweet ID or normalised text, including transitive connections. If any connected component has multiple labels, quarantine every row in it. Do not relabel or use confidence/majority vote to pick a winner.
4. Quarantine remaining connected components containing one ID linked to different normalised texts, as an identity-integrity problem.
5. For remaining repeated nonblank IDs, retain the first source record.
6. For remaining repeated normalised text, retain the first source record.

Choosing the earliest source record is a deterministic representative rule only after label agreement is established. Optional metadata from other copies is not merged. Missing optional metadata never causes exclusion. Blank IDs are reported but do not by themselves invalidate usable labelled text or link unrelated rows. Quarantined rows remain in the original data and the audit ledger for later review; exclusions can bias the class distribution.

Initial duplicate statistics overlap and must not be added together. The sequential removal ledger gives disjoint removal counts.

In [3]:
# Policy checks cover preserved sentiment cues and transitive conflicts.
assert normalise("  I can't!! 😡\tnever?  ") == "I can't!! 😡 never?"
assert normalise('e\u0301') == 'é'
assert normalise('Great!') != normalise('great!')
assert normalise('👩‍✈️?!') == '👩‍✈️?!'
fixture = pd.DataFrame({'tweet_id':['a','a','b',''],
                        'normalised_text':['one','two','two','other'],
                        'airline_sentiment':['negative','negative','positive','neutral']})
components = linked_groups(fixture)
assert components.iloc[0] == components.iloc[1] == components.iloc[2]
assert components.iloc[3] != components.iloc[0]
print('Normalisation and connected-group policy checks passed.')

Normalisation and connected-group policy checks passed.


In [4]:
summary, clean, distribution = run_audit(ROOT)
display(pd.read_csv(ROOT / 'results/audit/missing_values.csv', keep_default_na=False))
display(summary['initial_checks'])
display(pd.DataFrame(summary['removal_ledger']))
display(distribution)
print('Final usable labelled records:', summary['final_count'])
print('At least 10,000:', summary['at_least_10000'])
print('Conflicting connected groups:', summary['conflicting_components'])
print('Rows quarantined from conflicting groups:', next(s['removed'] for s in summary['removal_ledger']
      if s['step'] == 'quarantine_conflicting_label_components'))

,column,empty_cells,blank_including_whitespace
0,tweet_id,0,0
1,airline_sentiment,0,0
2,airline_sentiment_confidence,0,0
3,negativereason,5462,5462
4,negativereason_confidence,4118,4118
5,airline,0,0
6,airline_sentiment_gold,14600,14600
7,name,0,0
8,negativereason_gold,14608,14608
9,retweet_count,0,0


{'tweet_id': {'groups': 155,
  'rows_in_groups': 310,
  'excess_rows': 155,
  'conflicting_label_groups': 18},
 'raw_text': {'groups': 187,
  'rows_in_groups': 400,
  'excess_rows': 213,
  'conflicting_label_groups': 25},
 'normalised_text': {'groups': 189,
  'rows_in_groups': 405,
  'excess_rows': 216,
  'conflicting_label_groups': 26},
 'blank_text': 0,
 'invalid_label': 0,
 'blank_tweet_id': 0,
 'non_decimal_nonblank_tweet_id': 0}

,step,before,removed,remaining
0,blank_or_missing_text,14640,0,14640
1,invalid_or_missing_label,14640,0,14640
2,quarantine_conflicting_label_components,14640,59,14581
3,quarantine_inconsistent_id_text_components,14581,0,14581
4,duplicate_tweet_id_keep_first_source_row,14581,137,14444
5,duplicate_normalised_text_keep_first_source_row,14444,46,14398


,starting,final,final_percent
airline_sentiment,,,
negative,9178,9079,63.0574
neutral,3099,3044,21.1418
positive,2363,2275,15.8008


Final usable labelled records: 14398
At least 10,000: True
Conflicting connected groups: 26
Rows quarantined from conflicting groups: 59


## Investigating the earlier 14,250 figure

The following diagnostics compare raw ID uniqueness, raw text uniqueness, the chosen conservative normalisation and case-folding sensitivity. They are separate scenarios, not sequential removals. Case-folding is diagnostic only and is not used to edit or exclude the audited text. The earlier 14,250 result may have used different cleaning rules. No arbitrary extra removals are made to hit 14,250. Without the previous cleaning rules/notebook, the exact reason for the difference is unknown.

In [5]:
display(pd.Series(summary['count_investigation'], name='value').to_frame())
print('Difference from prior claim:', summary['final_count'] - 14250)
print('Remaining rows with sentiment confidence below 1:', summary['final_confidence_below_1'])
print('Remaining missing/nonnumeric confidence:', summary['final_confidence_missing_or_nonnumeric'])
print('Remaining confidence outside [0,1]:', summary['final_confidence_outside_0_1'])
print('No confidence threshold was used for exclusion.')

,value
all_original_rows,14640
unique_raw_tweet_id,14485
unique_raw_text,14427
unique_NFC_whitespace_text,14424
unique_NFC_whitespace_casefold_text,14406
chosen_policy_final,14398
previous_claim,14250
previous_claim_status,No prior notebook or rules provided. These dia...


Difference from prior claim: 148
Remaining rows with sentiment confidence below 1: 4111
Remaining missing/nonnumeric confidence: 0
Remaining confidence outside [0,1]: 0
No confidence threshold was used for exclusion.


## Remaining limitations and review points

- Near-duplicates remain: wording variants, altered URLs/mentions, case differences, retweet prefixes and semantically equivalent messages are not detected by this conservative exact key. A later split policy must consider leakage; no split is made now.
- Different IDs with identical text can be independent tweets. Collapsing them reduces repeated text weight but can remove legitimate repetition. Whitespace normalisation can also erase layout emphasis; original text is preserved.
- Conflicting labels may reflect ambiguity, context, sarcasm, mixed sentiment or annotation error. Quarantine is a transparent provisional choice, not a correction of ground truth. Confidence is not proof that a label is correct; no low-confidence row is discarded solely for that reason.
- Removing conflicts can change the population represented. Review `conflicting_groups.csv`, `removed_rows.csv` and the before/after class distribution. Optional metadata is incomplete and not needed for this text-label audit.
- Tweets are a historical, domain-specific sample; findings may not generalise to present-day airline customers, other languages or platforms. Class imbalance, short text, conversation context and annotation quality remain concerns. This audit does not independently re-annotate tweets or verify representativeness.
- The brief, marking scheme, lecturer materials, disclosure policy and prior audit are still needed. The student must understand the cleaning rules, justify exclusions, check sample groups and confirm requirements before Stage 2.

## Saved outputs

`data/processed/tweets_audited.csv` keeps original columns plus source record, normalised key and linked group. `results/audit/` contains JSON summary, missingness, duplicate/conflict groups, one-reason-per-row exclusions, sequential ledger, count diagnostics and class distribution. These CSV exports must also be read with `tweet_id` as a string. No model results exist.

In [6]:
# Independent output reconciliation and raw preservation checks.
removed = pd.read_csv(ROOT / 'results/audit/removed_rows.csv', dtype='string', keep_default_na=False)
saved = pd.read_csv(ROOT / 'data/processed/tweets_audited.csv', dtype='string', keep_default_na=False)
assert len(raw) == len(removed) + len(saved)
assert removed.source_row.is_unique and saved.source_row.is_unique
assert set(removed.source_row).isdisjoint(saved.source_row)
assert set(removed.source_row) | set(saved.source_row) == set(map(str, range(1, len(raw) + 1)))
assert sum(s['removed'] for s in summary['removal_ledger']) == len(removed)
assert int(distribution['final'].sum()) == len(saved)
original_lookup = raw.copy()
original_lookup.index = range(1, len(raw) + 1)
for col in raw.columns:
    assert saved[col].tolist() == original_lookup.loc[saved.source_row.astype(int), col].tolist(), col
assert hashlib.sha256((ROOT / 'data/original/Tweets.csv').read_bytes()).hexdigest() == EXPECTED_SHA256
print('All reconciliation, original-column preservation and checksum checks passed.')

All reconciliation, original-column preservation and checksum checks passed.
